# Construction and Visualization of Densest Subgraphs in Graphs
### CSAI 330 — Design and Analysis of Algorithms — Final Mini-Project

**Network category: Collaboration Networks** (Network Repository, networkrepository.com)

This notebook implements and compares four algorithms that incrementally construct the
**densest subgraph** of a network — the vertex subset maximizing edge density
|E(S)|/|S| (or, for one algorithm, triangle density) — across five real collaboration
(co-authorship) networks spanning small to very-large scale:

| Category | Algorithms | Approach |
|---|---|---|
| Greedy Peeling | **Charikar's algorithm**, **Greedy++** | Iteratively remove the lowest-(weighted-)degree vertex |
| Max-Flow-Based | **Goldberg's algorithm**, **Exact triangle-density algorithm** | Solve the DSP exactly via reductions to max-flow/min-cut |

For every (dataset, algorithm) pair we: (1) time the run, (2) render an incremental
"growth" video of the discovered densest subgraph being assembled, and (3) export
GraphML snapshots. We then compare all four algorithms' runtime growth vs. graph size
on one combined plot.

> This project's pipeline pattern (`.mtx` loading → per-algorithm snapshot recording →
> GraphML export → matplotlib/imageio video rendering → results table → runtime plot)
> is adapted from a classmate's separate shortest-path-algorithms project as a **structural**
> reference only; the network category (Collaboration Networks, not Economic & Trade),
> all four algorithms, and all analysis/code here are original to this project.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Video, Image, display

from data_io import DATASET_REGISTRY, load_dataset
from algorithms.charikar import charikar_peeling
from algorithms.greedy_pp import greedy_plus_plus
from algorithms.goldberg_maxflow import goldberg_densest_subgraph
from algorithms.triangle_density import exact_triangle_densest_subgraph
from algorithms.common import enumerate_triangles
import visualization as viz
import benchmark

## 1. Datasets

Five Collaboration Networks (co-authorship graphs), spanning small → very-large, as
required (team categories must pick ≥ half the team size in datasets; all five here
comfortably satisfy that for an 8–10 person team). Sizes below are the **real,
verified** node/edge counts of the files actually hosted on networkrepository.com today
(three of the five differ somewhat from the course PDF's illustrative table — see
`README.md`, "Datasets & limitations", for why; `ca-dblp-2012` is the real slug behind
the PDF's "ca-DBLP" row and matches its table exactly).

In [ ]:
for name, spec in DATASET_REGISTRY.items():
    print(f"{name:16s} tier={spec.tier:12s} expected n={spec.expected_nodes:>8} m={spec.expected_edges:>10}")

In [ ]:
graphs = {name: load_dataset(name) for name in DATASET_REGISTRY}
for name, G in graphs.items():
    print(f"{name:16s} n={G.number_of_nodes():>8} m={G.number_of_edges():>10}")

## 2. Algorithm 1 — Charikar's Greedy Peeling Algorithm

*M. Charikar, "Greedy Approximation Algorithms for Finding Dense Components in a
Graph", APPROX 2000.* Repeatedly removes the vertex of lowest weighted degree,
tracking density after each removal; returns the densest subgraph seen over the whole
peeling sequence. A 1/2-approximation to the true optimum, in O((n+m) log n) time.

**"Growing" visualization**: peeling *removes* vertices, but the assignment asks for the
construction to be shown *growing*. We exploit that the tail of the shrink sequence,
from the winning best-density snapshot down to the final singleton, is a strictly
increasing chain when read backwards — so we replay it in reverse: one vertex added at a
time, ending exactly at the discovered densest subgraph S\*.

In [ ]:
charikar_results = {}
for name, G in graphs.items():
    import time
    t0 = time.perf_counter()
    result = charikar_peeling(G)
    dt = time.perf_counter() - t0
    charikar_results[name] = result
    print(f"{name:16s} time={dt:8.3f}s  density={result['best_density']:.4f}  |S*|={len(result['best_nodes'])}")

In [ ]:
# Render one incremental-construction video + GraphML snapshots per dataset
os.makedirs('videos', exist_ok=True)
for name, G in graphs.items():
    result = charikar_results[name]
    frames = viz.growth_frames_from_peeling(result)
    viz.render_growth_video(G, frames, f'videos/{name}_Charikar.mp4', title=f'{name} — Charikar')
    viz.export_growth_snapshots(G, frames, f'outputs/{name}/Charikar')
    print(f'{name}: {len(frames)} frames rendered')

In [ ]:
# preview the smallest dataset's video inline
Video('videos/ca-netscience_Charikar.mp4', embed=True, width=500)

## 3. Algorithm 2 — Greedy++

*D. Boob, Y. Gao, R. Peng, S. Sawlani, C. Tsourakakis, D. Wang, J. Wang,
"Flowless: Extracting Densest Subgraphs Without Flow Computations" (WWW 2020).*
An iterative refinement of Charikar's peeling: each round re-peels the *entire* graph,
but ranks vertices by degree **plus** an accumulated "load" carried over from every
previous round's peeling. This discretized multiplicative-weights scheme provably
converges toward the exact LP-optimal density as the round count grows — narrowing
Charikar's 2x approximation gap without ever running a max-flow.

In [ ]:
greedy_pp_results = {}
for name, G in graphs.items():
    import time
    t0 = time.perf_counter()
    result = greedy_plus_plus(G)
    dt = time.perf_counter() - t0
    greedy_pp_results[name] = result
    print(f"{name:16s} time={dt:8.3f}s  density={result['best_density']:.4f}  "
          f"|S*|={len(result['best_nodes'])}  rounds={len(result['round_best_density'])}")

In [ ]:
for name, G in graphs.items():
    result = greedy_pp_results[name]
    frames = viz.growth_frames_from_greedy_pp(result)
    viz.render_growth_video(G, frames, f'videos/{name}_GreedyPP.mp4', title=f'{name} — Greedy++')
    viz.export_growth_snapshots(G, frames, f'outputs/{name}/GreedyPP')
    print(f'{name}: {len(frames)} frames rendered')

### Greedy++ convergence

`ca-dblp-2012` is the only dataset where Greedy++ ran more than a couple of rounds
before its density estimate stalled (patience-based early stop) — plotted below.

In [ ]:
benchmark.plot_greedy_pp_convergence(
    greedy_pp_results['ca-dblp-2012']['round_best_density'],
    'figures/greedy_pp_convergence.png', 'ca-dblp-2012')
Image('figures/greedy_pp_convergence.png')

## 4. Algorithm 3 — Goldberg's Exact Max-Flow Algorithm

*A. V. Goldberg, "Finding a Maximum Density Subgraph", UC Berkeley Tech Report
UCB/CSD-84-171, 1984.* Solves the densest subgraph problem **exactly** via parametric
max-flow/min-cut: binary search a candidate density `g`; build a flow network
(`s→v` capacity m, `v→t` capacity m + 2g − deg(v), `u↔v` capacity w(u,v) per edge);
a subgraph of density ≥ g exists iff the min-cut's source side is non-trivial. Converges
to the exact maximum density and its witness set.

**Sampling policy** (documented, reproducible — see `src/benchmark.py` and `README.md`):
runs at **full size** on ca-netscience/ca-CSphd/ca-GrQc/ca-HepTh (empirically verified
tractable — ca-HepTh's full 11,204-node/117,619-edge network completes in ~2 minutes);
on `ca-dblp-2012` it uses a BFS/snowball sample (seed=42, ~10,000 nodes) so the search
finishes in minutes instead of being intractable at 317K nodes.

In [ ]:
from data_io import sample_subgraph
goldberg_results = {}
goldberg_graphs = {}
for name, G in graphs.items():
    G_variant, sampled = benchmark._prepare_variant(G, name, benchmark.GOLDBERG_SAMPLE_TARGET_N)
    goldberg_graphs[name] = G_variant
    result = goldberg_densest_subgraph(G_variant)
    goldberg_results[name] = result
    flag = ' (BFS-sampled)' if sampled else ''
    print(f"{name:16s}{flag:16s} n={G_variant.number_of_nodes():>7} search_time={result['search_time']:8.3f}s "
          f"density={result['best_density']:.4f}  |S*|={len(result['best_nodes'])}")

In [ ]:
for name, G_variant in goldberg_graphs.items():
    result = goldberg_results[name]
    frames = viz.growth_frames_from_goldberg(result, G_variant)
    viz.render_growth_video(G_variant, frames, f'videos/{name}_Goldberg.mp4', title=f'{name} — Goldberg')
    viz.export_growth_snapshots(G_variant, frames, f'outputs/{name}/Goldberg')
    print(f'{name}: {len(frames)} frames rendered')

## 5. Algorithm 4 — Exact Triangle-Density Algorithm

Generalizes Goldberg's reduction to a 3-uniform hypergraph term (per S. Khuller & B. Saha,
"On Finding Dense Subgraphs", ICALP 2009; motivated by C. Tsourakakis et al.'s
triangle-densest-subgraph problem, KDD 2013 / WWW 2015): each enumerated triangle gets an
auxiliary flow node (`s→Δ` capacity 1, `Δ→{u,v,w}` capacity ∞), and `v→t` capacity `g`
per vertex; solved by the same parametric binary search.

**Sampling policy**: the flow network's size scales with the **triangle count**, not the
node count, so this needs a much more aggressive, per-dataset sample: ca-netscience and
ca-CSphd run at full size (both have only a handful of triangles); ca-GrQc (47,779
triangles at full size), ca-HepTh (**3.36 million** triangles at full size — empirically
confirmed intractable within minutes), and ca-dblp-2012 are each BFS-sampled to a size
chosen so the sampled subgraph's triangle count stays in the low thousands.

In [ ]:
triangle_results = {}
triangle_graphs = {}
for name, G in graphs.items():
    G_variant, sampled = benchmark._prepare_variant(G, name, benchmark.TRIANGLE_SAMPLE_TARGET_N)
    triangle_graphs[name] = G_variant
    result = exact_triangle_densest_subgraph(G_variant)
    triangle_results[name] = result
    flag = ' (BFS-sampled)' if sampled else ''
    print(f"{name:16s}{flag:16s} n={G_variant.number_of_nodes():>7} triangles={result['n_triangles_total']:>8} "
          f"search_time={result['search_time']:8.3f}s density={result['best_density']:.4f} |S*|={len(result['best_nodes'])}")

In [ ]:
for name, G_variant in triangle_graphs.items():
    result = triangle_results[name]
    if result['n_triangles_total'] == 0:
        print(f'{name}: no triangles, skipping video')
        continue
    triangles = enumerate_triangles(G_variant)
    frames = viz.growth_frames_from_triangle_density(result, G_variant, triangles)
    viz.render_growth_video(G_variant, frames, f'videos/{name}_TriangleDensity.mp4', title=f'{name} — TriangleDensity')
    viz.export_growth_snapshots(G_variant, frames, f'outputs/{name}/TriangleDensity')
    print(f'{name}: {len(frames)} frames rendered')

## 6. Results Summary

In [ ]:
results_df = pd.read_csv('figures/benchmark_results.csv')
results_df

## 7. Runtime Growth Analysis

Empirical computational cost growth vs. nodes and vs. edges, all four algorithms on one
combined plot with a legend (log-log scale, given the multi-order-of-magnitude size
range from 379 to 317,080 nodes).

In [ ]:
display(Image('figures/runtime_vs_nodes.png'))
display(Image('figures/runtime_vs_edges.png'))

## 8. Conclusion & Limitations

- **Charikar** and **Greedy++** scale near-linearly and ran at full size on every
  dataset including the 317K-node `ca-dblp-2012`; Greedy++ consistently matches or
  slightly beats Charikar's density, as expected from the theory.
- **Goldberg's algorithm** is exact and, on every dataset checked, its density matches
  or exceeds both greedy heuristics (as it must) — on `ca-dblp-2012` its BFS-sampled run
  actually recovered the *same* optimal subgraph Greedy++ found on the full graph, a nice
  cross-validation of both the sampling approach and the algorithms' correctness.
- **Exact triangle-density** required the most aggressive sampling, since its flow
  network scales with triangle count rather than node/edge count — `ca-HepTh`'s full
  3.36-million triangles made the full-size flow network intractable, so it (along with
  `ca-GrQc` and `ca-dblp-2012`) uses a documented, reproducible BFS sample sized to keep
  the triangle count in the low thousands.
- All correctness claims are backed by a 39-test pytest suite (`tests/`), including exact
  agreement between Goldberg's algorithm and brute-force ground truth on tiny synthetic
  fixtures, and cross-algorithm consistency checks (the exact algorithm must never be
  beaten by either heuristic).

See `README.md` for the full write-up (category & applications, algorithm details,
dataset table, and limitations).